***

### Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'TIMS Data')
path_main = os.path.join(path_sp, 'Data')
path_out  = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Safety')
path_tims = os.path.join(path_out, 'TIMS')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'TIMS')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'TIMS')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

### Importing Raw TIMS Data Downloads

***

In [ ]:
year_start = 2012
year_end = 2021
years_to_import = range(year_start, year_end+1)

In [ ]:
categories = [cat for cat in os.listdir(path_tims) if '.xlsx' not in cat]
categories



#### Jurisdictions

In [ ]:

dict_cat1 = {}

for cat in categories:
    path_cat = os.path.join(path_tims, cat, 'Jurisdictions')

    dict_counties = {}
    for county in os.listdir(path_cat):
        path_county = os.path.join(path_cat, county)
    
        list_files = []
        for file in os.listdir(path_county):
            list_files.append(file)
            
        dict_counties[county] = list_files
        
    dict_cat1[cat] = dict_counties

dict_cat1


In [ ]:
print('Organizing raw TIMS data...')
print('')


list_df_final = []
for cat in dict_cat1.keys():
    print(cat)

    list_df_total = []
    list_df_mvmt  = []
    list_df_non1  = []
    list_df_non2  = []
    
    for county in dict_cat1[cat]:
        print(county)
        for file in tqdm(dict_cat1[cat][county]):
            path_csv = os.path.join(path_tims, cat, 'Jurisdictions', county, file)
            df_temp = pd.read_csv(path_csv)
            
            df_temp = df_temp.iloc[:,0:3]

            metric = re.sub('-', ' ', re.sub(r'-[^-]*$', '', file))
            metric = metric.capitalize()
            df_temp.columns = ['Year', metric, metric + '_5 Year Rolling Average']

            df_temp = df_temp[df_temp['Year'].isin(years_to_import)]

            jurisdiction = re.sub('.csv', '', re.sub('.*-', '', file))
            jurisdiction = jurisdiction.capitalize()
            if jurisdiction == 'Unincorporated':
                jurisdiction = county + ' Unincorporated'
            
            df_temp['County'      ] = county
            df_temp['Jurisdiction'] = jurisdiction

            df_temp = df_temp.set_index(['County', 'Jurisdiction', 'Year']).reset_index()

            if cat == 'Non-Motorized':
                if 'serious' in file:
                    list_df_non2.append(df_temp)
                else:
                    list_df_non1.append(df_temp)
            else:
                if 'mvm' in file:
                    list_df_mvmt.append(df_temp)
                else:
                    list_df_total.append(df_temp)
    if cat == 'Non-Motorized':                
        df_non1  = pd.concat(list_df_non1 )
        df_non2  = pd.concat(list_df_non2)
        list_df_cat = [df_non1, df_non2]
    else:                
        df_mvmt  = pd.concat(list_df_mvmt )
        df_total = pd.concat(list_df_total)
        list_df_cat = [df_total, df_mvmt]

    df_cat = ft.reduce(lambda left, right: pd.merge(left, right, on = ['County', 'Jurisdiction', 'Year']), list_df_cat)
    list_df_final.append(df_cat)

df_tims1 = ft.reduce(lambda left, right: pd.merge(left, right, on = ['County', 'Jurisdiction', 'Year']), list_df_final)
df_tims1 = df_tims1.sort_values(['County', 'Jurisdiction', 'Year'], ascending = [True, True, True])

print('')
print('Finished !!')
print('')

df_tims1.head()

#### Counties

In [ ]:

dict_cat2 = {}

for cat in categories:
    path_cat = os.path.join(path_tims, cat, 'Counties')

    list_files = []
    for file in os.listdir(path_cat):
        path_counties = os.path.join(path_cat, file)
        list_files.append(path_counties)
    dict_cat2[cat] = list_files

dict_cat2


In [ ]:
print('Organizing raw TIMS data...')
print('')


list_df_final = []
for cat in dict_cat2.keys():
    print(cat)

    list_df_total = []
    list_df_mvmt  = []
    list_df_non1  = []
    list_df_non2  = []
    
    for path_csv in tqdm(dict_cat2[cat]):
        df_temp = pd.read_csv(path_csv)
        file = re.sub(r'.*\\', '', path_csv)
        
        df_temp = df_temp.iloc[:,0:3]

        metric = re.sub('-', ' ', re.sub(r'-[^-]*$', '', file))
        metric = metric.capitalize()
        df_temp.columns = ['Year', metric, metric + '_5 Year Rolling Average']

        df_temp = df_temp[df_temp['Year'].isin(years_to_import)]

        county = re.sub('.csv', '', re.sub(r'.*-', '', file))
        county = county.title()
        
        df_temp['County'] = county

        df_temp = df_temp.set_index(['County', 'Year']).reset_index()

        if cat == 'Non-Motorized':
            if 'serious' in file:
                list_df_non2.append(df_temp)
            else:
                list_df_non1.append(df_temp)
        else:
            if 'mvm' in file:
                list_df_mvmt.append(df_temp)
            else:
                list_df_total.append(df_temp)
    if cat == 'Non-Motorized':                
        df_non1  = pd.concat(list_df_non1 )
        df_non2  = pd.concat(list_df_non2)
        list_df_cat = [df_non1, df_non2]
    else:                
        df_mvmt  = pd.concat(list_df_mvmt )
        df_total = pd.concat(list_df_total)
        list_df_cat = [df_total, df_mvmt]

    df_cat = ft.reduce(lambda left, right: pd.merge(left, right, on = ['County', 'Year']), list_df_cat)
    list_df_final.append(df_cat)

df_tims2 = ft.reduce(lambda left, right: pd.merge(left, right, on = ['County', 'Year']), list_df_final)
df_tims2 = df_tims2.sort_values(['County', 'Year'], ascending = [True, True])

print('')
print('Finished !!')
print('')

df_tims2.head()

#### MPO

In [ ]:

dict_cat3 = {}

for cat in categories:
    path_cat = os.path.join(path_tims, cat, 'MPO')

    list_files = []
    for file in os.listdir(path_cat):
        path_mpo = os.path.join(path_cat, file)
        list_files.append(path_mpo)
    dict_cat3[cat] = list_files

dict_cat3


In [ ]:
print('Organizing raw TIMS data...')
print('')


list_df_final = []
for cat in dict_cat3.keys():
    print(cat)

    list_df_total = []
    list_df_mvmt  = []
    list_df_non1  = []
    list_df_non2  = []
    
    for path_csv in tqdm(dict_cat3[cat]):
        df_temp = pd.read_csv(path_csv)
        file = re.sub(r'.*\\', '', path_csv)
        
        df_temp = df_temp.iloc[:,0:3]

        metric = re.sub('-', ' ', re.sub(r'-[^-]*$', '', file))
        metric = metric.capitalize()
        df_temp.columns = ['Year', metric, metric + '_5 Year Rolling Average']

        df_temp = df_temp[df_temp['Year'].isin(years_to_import)]

        mpo = re.sub('.csv', '', re.sub(r'.*-', '', file))
        
        df_temp['MPO'] = mpo

        df_temp = df_temp.set_index(['MPO', 'Year']).reset_index()

        if cat == 'Non-Motorized':
            if 'serious' in file:
                list_df_non2.append(df_temp)
            else:
                list_df_non1.append(df_temp)
        else:
            if 'mvm' in file:
                list_df_mvmt.append(df_temp)
            else:
                list_df_total.append(df_temp)
    if cat == 'Non-Motorized':                
        df_non1  = pd.concat(list_df_non1 )
        df_non2  = pd.concat(list_df_non2)
        list_df_cat = [df_non1, df_non2]
    else:                
        df_mvmt  = pd.concat(list_df_mvmt )
        df_total = pd.concat(list_df_total)
        list_df_cat = [df_total, df_mvmt]

    df_cat = ft.reduce(lambda left, right: pd.merge(left, right, on = ['MPO', 'Year']), list_df_cat)
    list_df_final.append(df_cat)

df_tims3 = ft.reduce(lambda left, right: pd.merge(left, right, on = ['MPO', 'Year']), list_df_final)
df_tims3 = df_tims3.sort_values(['MPO', 'Year'], ascending = [True, True])

print('')
print('Finished !!')
print('')

df_tims3.head()

#### Statewide

In [ ]:

dict_cat4 = {}

for cat in categories:
    path_cat = os.path.join(path_tims, cat, 'Statewide')

    list_files = []
    for file in os.listdir(path_cat):
        path_mpo = os.path.join(path_cat, file)
        list_files.append(path_mpo)        
    dict_cat4[cat] = list_files

dict_cat4


In [ ]:
print('Organizing raw TIMS data...')
print('')


list_df_final = []
for cat in dict_cat4.keys():
    print(cat)

    list_df_total = []
    list_df_mvmt  = []
    list_df_non1  = []
    list_df_non2  = []
    
    for path_csv in tqdm(dict_cat4[cat]):
        df_temp = pd.read_csv(path_csv)
        file = re.sub(r'.*\\', '', path_csv)
        
        df_temp = df_temp.iloc[:,0:3]

        metric = re.sub('-', ' ', re.sub(r'-[^-]*$', '', file))
        metric = metric.capitalize()
        df_temp.columns = ['Year', metric, metric + '_5 Year Rolling Average']

        df_temp = df_temp[df_temp['Year'].isin(years_to_import)]

        state = re.sub('.csv', '', re.sub(r'.*-', '', file))
        
        df_temp['State'] = state

        df_temp = df_temp.set_index(['State', 'Year']).reset_index()

        if cat == 'Non-Motorized':
            if 'serious' in file:
                list_df_non2.append(df_temp)
            else:
                list_df_non1.append(df_temp)
        else:
            if 'mvm' in file:
                list_df_mvmt.append(df_temp)
            else:
                list_df_total.append(df_temp)
    if cat == 'Non-Motorized':                
        df_non1  = pd.concat(list_df_non1 )
        df_non2  = pd.concat(list_df_non2)
        list_df_cat = [df_non1, df_non2]
    else:                
        df_mvmt  = pd.concat(list_df_mvmt )
        df_total = pd.concat(list_df_total)
        list_df_cat = [df_total, df_mvmt]

    df_cat = ft.reduce(lambda left, right: pd.merge(left, right, on = ['State', 'Year']), list_df_cat)
    list_df_final.append(df_cat)

df_tims4 = ft.reduce(lambda left, right: pd.merge(left, right, on = ['State', 'Year']), list_df_final)
df_tims4 = df_tims4.sort_values(['State', 'Year'], ascending = [True, True])

print('')
print('Finished !!')
print('')

df_tims4.head()

***

### Organizing

***

In [ ]:
df_tims1.columns

In [ ]:

# Subset all TIMS data into different categories
list_id = ['County', 'Jurisdiction', 'Year']

df_tims1_fat = df_tims1[list_id + [              'Fatalities',               'Fatalities_5 Year Rolling Average']]
df_tims1_ser = df_tims1[list_id + [        'Serious injuries',         'Serious injuries_5 Year Rolling Average']]
df_tims1_non = df_tims1[list_id + ['Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                 , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]

df_tims1_fat100 = df_tims1[list_id + [      'Fatalities 100 mvmt',      'Fatalities 100 mvmt_5 Year Rolling Average']]
df_tims1_ser100 = df_tims1[list_id + ['Serious injuries 100 mvm' , 'Serious injuries 100 mvm_5 Year Rolling Average']]


# Set columns of jurisdictions by counties
df_tims1_cols = df_tims1_fat.pivot_table(index = 'Year'
                                        , columns = ['County', 'Jurisdiction']
                                        , values = ['Fatalities', 'Fatalities_5 Year Rolling Average']).reset_index()
cols = [col[2] for col in df_tims1_cols.columns][1:]
cols = ['Year'] + cols


# Fatalities and Fatalities 100/MVMT
df_tims1_fat_2      = df_tims1_fat   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities'                                ).reset_index()
df_tims1_fat_2_5    = df_tims1_fat   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities_5 Year Rolling Average'         ).reset_index()
df_tims1_fat100_2   = df_tims1_fat100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities 100 mvmt'                       ).reset_index()
df_tims1_fat100_2_5 = df_tims1_fat100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities 100 mvmt_5 Year Rolling Average').reset_index()


# Serious Injuries and Serious Injuries 100/MVMT
df_tims1_ser_2      = df_tims1_ser   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries'                               ).reset_index()
df_tims1_ser_2_5    = df_tims1_ser   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries_5 Year Rolling Average'        ).reset_index()
df_tims1_ser100_2   = df_tims1_ser100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries 100 mvm'                       ).reset_index()
df_tims1_ser100_2_5 = df_tims1_ser100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries 100 mvm_5 Year Rolling Average').reset_index()


# Non-Motorized Fatalities and Serious Injuries
df_tims1_non_fat_2   = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized fatalities'                       ).reset_index()
df_tims1_non_fat_2_5 = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized fatalities_5 Year Rolling Average').reset_index()
df_tims1_non_si_2    = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized serious in'                       ).reset_index()
df_tims1_non_si_2_5  = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized serious in_5 Year Rolling Average').reset_index()

# Reorganize columns
df_tims1_fat_2       = df_tims1_fat_2      [cols]
df_tims1_fat_2_5     = df_tims1_fat_2_5    [cols]
df_tims1_fat100_2    = df_tims1_fat100_2   [cols]
df_tims1_fat100_2_5  = df_tims1_fat100_2_5 [cols]
df_tims1_ser_2       = df_tims1_ser_2      [cols]
df_tims1_ser_2_5     = df_tims1_ser_2_5    [cols]
df_tims1_ser100_2    = df_tims1_ser100_2   [cols]
df_tims1_ser100_2_5  = df_tims1_ser100_2_5 [cols]
df_tims1_non_fat_2   = df_tims1_non_fat_2  [cols]
df_tims1_non_fat_2_5 = df_tims1_non_fat_2_5[cols]
df_tims1_non_si_2    = df_tims1_non_si_2   [cols]
df_tims1_non_si_2_5  = df_tims1_non_si_2_5 [cols]


## Jurisdictions
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data by Jurisdiction.xlsx'), engine='xlsxwriter') as writer:
#     df_tims1           .to_excel(writer, index = False, sheet_name = 'All'                            )
#     df_tims_fat_2      .to_excel(writer, index = False, sheet_name = 'Fatalities'                     )
#     df_tims_fat_2_5    .to_excel(writer, index = False, sheet_name = 'Fatalities 5 year'              )
#     df_tims_fat100_2   .to_excel(writer, index = False, sheet_name = 'Fatalities rate'                )
#     df_tims_fat100_2_5 .to_excel(writer, index = False, sheet_name = 'Fatalities rate 5 year'         )
#     df_tims_ser_2      .to_excel(writer, index = False, sheet_name = 'Serious injuries'               )
#     df_tims_ser_2_5    .to_excel(writer, index = False, sheet_name = 'Serious injuries 5 year'        )
#     df_tims_ser100_2   .to_excel(writer, index = False, sheet_name = 'Serious injuries rate'          )
#     df_tims_ser100_2_5 .to_excel(writer, index = False, sheet_name = 'Serious injuries rate 5 year'   )
#     df_tims_non_fat_2  .to_excel(writer, index = False, sheet_name = 'Non motorized fatalities'       )
#     df_tims_non_fat_2_5.to_excel(writer, index = False, sheet_name = 'Non motorized fatalities 5 year')
#     df_tims_non_si_2   .to_excel(writer, index = False, sheet_name = 'Non motorized serious in'       )
#     df_tims_non_si_2_5 .to_excel(writer, index = False, sheet_name = 'Non motorized serious in 5 year')


In [ ]:
df_tims2.columns

In [ ]:

# Subset all TIMS data into different categories
list_id = ['County', 'Year']

df_tims2_fat = df_tims2[list_id + [              'Fatalities',               'Fatalities_5 Year Rolling Average']]
df_tims2_ser = df_tims2[list_id + [        'Serious injuries',         'Serious injuries_5 Year Rolling Average']]
df_tims2_non = df_tims2[list_id + ['Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                 , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]

df_tims2_fat100 = df_tims2[list_id + [      'Fatalities 100 mvmt',      'Fatalities 100 mvmt_5 Year Rolling Average']]
df_tims2_ser100 = df_tims2[list_id + ['Serious injuries 100 mvm' , 'Serious injuries 100 mvm_5 Year Rolling Average']]


# Set columns of jurisdictions by counties
df_tims2_cols = df_tims2_fat.pivot_table(index = 'Year'
                                        , columns = 'County'
                                        , values = ['Fatalities', 'Fatalities_5 Year Rolling Average']).reset_index()
cols = [col[1] for col in df_tims2_cols.columns][1:]
cols = ['Year'] + cols


# Fatalities and Fatalities 100/MVMT
df_tims2_fat_2      = df_tims2_fat   .pivot_table(index = 'Year', columns = 'County', values = 'Fatalities'                                ).reset_index()
df_tims2_fat_2_5    = df_tims2_fat   .pivot_table(index = 'Year', columns = 'County', values = 'Fatalities_5 Year Rolling Average'         ).reset_index()
df_tims2_fat100_2   = df_tims2_fat100.pivot_table(index = 'Year', columns = 'County', values = 'Fatalities 100 mvmt'                       ).reset_index()
df_tims2_fat100_2_5 = df_tims2_fat100.pivot_table(index = 'Year', columns = 'County', values = 'Fatalities 100 mvmt_5 Year Rolling Average').reset_index()


# Serious Injuries and Serious Injuries 100/MVMT
df_tims2_ser_2      = df_tims2_ser   .pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries'                               ).reset_index()
df_tims2_ser_2_5    = df_tims2_ser   .pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries_5 Year Rolling Average'        ).reset_index()
df_tims2_ser100_2   = df_tims2_ser100.pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries 100 mvm'                       ).reset_index()
df_tims2_ser100_2_5 = df_tims2_ser100.pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries 100 mvm_5 Year Rolling Average').reset_index()


# Non-Motorized Fatalities and Serious Injuries
df_tims2_non_fat_2   = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized fatalities'                       ).reset_index()
df_tims2_non_fat_2_5 = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized fatalities_5 Year Rolling Average').reset_index()
df_tims2_non_si_2    = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized serious in'                       ).reset_index()
df_tims2_non_si_2_5  = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized serious in_5 Year Rolling Average').reset_index()

# Reorganize columns
df_tims2_fat_2       = df_tims2_fat_2      [cols]
df_tims2_fat_2_5     = df_tims2_fat_2_5    [cols]
df_tims2_fat100_2    = df_tims2_fat100_2   [cols]
df_tims2_fat100_2_5  = df_tims2_fat100_2_5 [cols]
df_tims2_ser_2       = df_tims2_ser_2      [cols]
df_tims2_ser_2_5     = df_tims2_ser_2_5    [cols]
df_tims2_ser100_2    = df_tims2_ser100_2   [cols]
df_tims2_ser100_2_5  = df_tims2_ser100_2_5 [cols]
df_tims2_non_fat_2   = df_tims2_non_fat_2  [cols]
df_tims2_non_fat_2_5 = df_tims2_non_fat_2_5[cols]
df_tims2_non_si_2    = df_tims2_non_si_2   [cols]
df_tims2_non_si_2_5  = df_tims2_non_si_2_5 [cols]


## Counties
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data by County.xlsx'), engine='xlsxwriter') as writer:
#     df_tims2            .to_excel(writer, index = False, sheet_name = 'All'                            )
#     df_tims2_fat_2      .to_excel(writer, index = False, sheet_name = 'Fatalities'                     )
#     df_tims2_fat_2_5    .to_excel(writer, index = False, sheet_name = 'Fatalities 5 year'              )
#     df_tims2_fat100_2   .to_excel(writer, index = False, sheet_name = 'Fatalities rate'                )
#     df_tims2_fat100_2_5 .to_excel(writer, index = False, sheet_name = 'Fatalities rate 5 year'         )
#     df_tims2_ser_2      .to_excel(writer, index = False, sheet_name = 'Serious injuries'               )
#     df_tims2_ser_2_5    .to_excel(writer, index = False, sheet_name = 'Serious injuries 5 year'        )
#     df_tims2_ser100_2   .to_excel(writer, index = False, sheet_name = 'Serious injuries rate'          )
#     df_tims2_ser100_2_5 .to_excel(writer, index = False, sheet_name = 'Serious injuries rate 5 year'   )
#     df_tims2_non_fat_2  .to_excel(writer, index = False, sheet_name = 'Non motorized fatalities'       )
#     df_tims2_non_fat_2_5.to_excel(writer, index = False, sheet_name = 'Non motorized fatalities 5 year')
#     df_tims2_non_si_2   .to_excel(writer, index = False, sheet_name = 'Non motorized serious in'       )
#     df_tims2_non_si_2_5 .to_excel(writer, index = False, sheet_name = 'Non motorized serious in 5 year')


In [ ]:
# # MPO
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data by MPO.xlsx'), engine='xlsxwriter') as writer:
#     df_tims3.to_excel(writer, index = False, sheet_name = 'All')

# # Statewide
# with pd.ExcelWriter(os.path.join(path_tims, 'TIMS SWITRS Data Statewide.xlsx'), engine='xlsxwriter') as writer:
#     df_tims4.to_excel(writer, index = False, sheet_name = 'Statewide')

***

## Safety Indicators

***

In [ ]:

# Safety_1 Collision Rates
df_safety1_1 = df_tims1[['County', 'Jurisdiction', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_2 = df_tims2[['County', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_3 = df_tims3[['MPO', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                      , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_4 = df_tims4[['State', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                        , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Safety_2 Non-Motorized
df_safety2_1 = df_tims1[['County', 'Jurisdiction', 'Year', 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety2_2 = df_tims2[['County', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                         , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_3 = df_tims3[['MPO', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                      , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_4 = df_tims4[['State', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                        , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]


In [ ]:

# # Safety_1 Collision Rates
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 Jurisdictions TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_safety1_1.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 Counties TIMS.xlsx'     ), engine='xlsxwriter') as writer:
#     df_safety1_2.to_excel(writer, index = False, sheet_name = 'Counties'     )
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 MPO TIMS.xlsx'          ), engine='xlsxwriter') as writer:
#     df_safety1_3.to_excel(writer, index = False, sheet_name = 'MPO'          )
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_1 Collision Rates', 'Safety_1 Statewide TIMS.xlsx'    ), engine='xlsxwriter') as writer:
#     df_safety1_4.to_excel(writer, index = False, sheet_name = 'Statewide'    )

# # Safety_2 Non-Motorized
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 Jurisdictions TIMS.xlsx'), engine='xlsxwriter') as writer:
#     df_safety2_1.to_excel(writer, index = False, sheet_name = 'Jurisdictions')
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 Counties TIMS.xlsx'     ), engine='xlsxwriter') as writer:
#     df_safety2_2.to_excel(writer, index = False, sheet_name = 'Counties'     )
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 MPO TIMS.xlsx'          ), engine='xlsxwriter') as writer:
#     df_safety2_3.to_excel(writer, index = False, sheet_name = 'MPO'          )
# with pd.ExcelWriter(os.path.join(path_out, 'Safety_2 Non-Motorized', 'Safety_2 Statewide TIMS.xlsx'    ), engine='xlsxwriter') as writer:
#     df_safety2_4.to_excel(writer, index = False, sheet_name = 'Statewide'    )


In [ ]:
df_safety1_3['State'] = 'CA'
df_safety1_4['MPO'  ] = 'Statewide'

df_safety_1 = pd.concat([df_safety1_3, df_safety1_4])
df_safety_1 = df_safety_1.drop('State', axis = 1)
df_safety_1 = df_safety_1.rename(columns = {'MPO':'Group'})

df_safety2_3['State'] = 'CA'
df_safety2_4['MPO'  ] = 'Statewide'

df_safety_2 = pd.concat([df_safety2_3, df_safety2_4])
df_safety_2 = df_safety_2.drop('State', axis = 1)
df_safety_2 = df_safety_2.rename(columns = {'MPO':'Group'})

df_safety_2

In [ ]:


df_safety_1.to_csv(os.path.join(path_agol, 'Safety_1', 'Safety_1 MPO TIMS.csv'      ), index = False)
df_safety_2.to_csv(os.path.join(path_agol, 'Safety_2', 'Safety_2 MPO TIMS.csv'      ), index = False)
